# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [4]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_ollama.embeddings import OllamaEmbeddings


## 1. Set Up Paths and the Vector Store

In [5]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

PosixPath('/Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/Advanced_Rag_Codes-main/04_vector_stores')

In [7]:
# # Load environment variables from the local .env file.
# dotenv_path = project_root / ".env"
# load_dotenv(dotenv_path=dotenv_path)

# if not os.getenv("OPENAI_API_KEY"):
#     raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

# print(f"Loaded environment from: {dotenv_path}")

In [8]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: /Users/itaakash/Desktop/Lakhan/Tasks/ragAdancedCampusX/Advanced_Rag_Codes-main/04_vector_stores/notebooks/chroma_langchain_db


In [9]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [10]:
# Create the embedding model and connect it to a persistent Chroma store.
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = OllamaEmbeddings(model="embeddinggemma")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [11]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [12]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [13]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [16]:
print(uuid4())

1e0e86db-c2e0-43ef-8feb-156a72d659ca


In [17]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=8cdb3a2e-2bb4-43ae-8efe-0faa846a7211
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=03cfc174-e179-4be0-ab37-ac473cf5d973
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=6fd46545-28fc-4a3d-bd97-e69e351e4202
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=2f184dfb-bf53-403d-845f-4f6241f19147
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=183020ce-f764-46c2-afc8-8bccf31ce62f
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [18]:
documents[0].id

'8cdb3a2e-2bb4-43ae-8efe-0faa846a7211'

In [20]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
8cdb3a2e-2bb4-43ae-8efe-0faa846a7211
03cfc174-e179-4be0-ab37-ac473cf5d973
6fd46545-28fc-4a3d-bd97-e69e351e4202
4e8de225-921f-4ee3-82d5-721f41f8cd8d
2f184dfb-bf53-403d-845f-4f6241f19147
183020ce-f764-46c2-afc8-8bccf31ce62f
2bea6d38-e484-4e0f-9264-03504dd21938
06db3fca-6b13-40ce-81d4-4499e0567785
7efecb79-06bb-4a6d-86cc-0a1ac51514e3
15ad8bae-03c7-4277-a9f4-dd754c55b69c

Total inserted documents: 10


## 4. Read the Stored Data Back

In [21]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [36]:
raw_records["metadatas"]

[{'doc_number': 1, 'topic': 'AI'},
 {'doc_number': 2, 'topic': 'AI'},
 {'doc_number': 3, 'topic': 'AI'},
 {'doc_number': 4, 'topic': 'RAG'},
 {'doc_number': 5, 'topic': 'RAG'},
 {'doc_number': 6, 'topic': 'RAG'},
 {'doc_number': 7, 'topic': 'LLM'},
 {'topic': 'LLM', 'doc_number': 8},
 {'doc_number': 9, 'topic': 'Cricket'},
 {'topic': 'Cricket', 'doc_number': 10}]

In [37]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [38]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
8cdb3a2e-2bb4-43ae-8efe-0faa846a7211
03cfc174-e179-4be0-ab37-ac473cf5d973
6fd46545-28fc-4a3d-bd97-e69e351e4202


In [39]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['06db3fca-6b13-40ce-81d4-4499e0567785',
 '7efecb79-06bb-4a6d-86cc-0a1ac51514e3',
 '15ad8bae-03c7-4277-a9f4-dd754c55b69c']

In [40]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=06db3fca-6b13-40ce-81d4-4499e0567785
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=7efecb79-06bb-4a6d-86cc-0a1ac51514e3
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=15ad8bae-03c7-4277-a9f4-dd754c55b69c
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [41]:
print(selected_documents)

[Document(id='06db3fca-6b13-40ce-81d4-4499e0567785', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='7efecb79-06bb-4a6d-86cc-0a1ac51514e3', metadata={'doc_number': 9, 'topic': 'Cricket'}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='15ad8bae-03c7-4277-a9f4-dd754c55b69c', metadata={'doc_number': 10, 'topic': 'Cricket'}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [42]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [43]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=06db3fca-6b13-40ce-81d4-4499e0567785
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=2f184dfb-bf53-403d-845f-4f6241f19147
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [44]:
search_results

[Document(id='4e8de225-921f-4ee3-82d5-721f41f8cd8d', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='06db3fca-6b13-40ce-81d4-4499e0567785', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='2f184dfb-bf53-403d-845f-4f6241f19147', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [45]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='4e8de225-921f-4ee3-82d5-721f41f8cd8d', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.7007133960723877),
 (Document(id='06db3fca-6b13-40ce-81d4-4499e0567785', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  1.0090618133544922),
 (Document(id='2f184dfb-bf53-403d-845f-4f6241f19147', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0321404933929443),
 (Document(id='2bea6d38-e484-4e0f-9264-03504dd21938', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.2556946277618408)]

## 6. Update Existing Documents

In [46]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['4e8de225-921f-4ee3-82d5-721f41f8cd8d',
 '06db3fca-6b13-40ce-81d4-4499e0567785']

In [47]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=06db3fca-6b13-40ce-81d4-4499e0567785
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [48]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [49]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
4e8de225-921f-4ee3-82d5-721f41f8cd8d
06db3fca-6b13-40ce-81d4-4499e0567785


In [50]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=06db3fca-6b13-40ce-81d4-4499e0567785
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [51]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [52]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=4e8de225-921f-4ee3-82d5-721f41f8cd8d
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=2f184dfb-bf53-403d-845f-4f6241f19147
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [53]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['7efecb79-06bb-4a6d-86cc-0a1ac51514e3',
 '15ad8bae-03c7-4277-a9f4-dd754c55b69c']

In [54]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
7efecb79-06bb-4a6d-86cc-0a1ac51514e3
15ad8bae-03c7-4277-a9f4-dd754c55b69c


In [55]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
8cdb3a2e-2bb4-43ae-8efe-0faa846a7211
03cfc174-e179-4be0-ab37-ac473cf5d973
6fd46545-28fc-4a3d-bd97-e69e351e4202
4e8de225-921f-4ee3-82d5-721f41f8cd8d
2f184dfb-bf53-403d-845f-4f6241f19147
183020ce-f764-46c2-afc8-8bccf31ce62f
2bea6d38-e484-4e0f-9264-03504dd21938
06db3fca-6b13-40ce-81d4-4499e0567785

Deleted ids still present?
7efecb79-06bb-4a6d-86cc-0a1ac51514e3: False
15ad8bae-03c7-4277-a9f4-dd754c55b69c: False


In [56]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
